# 🦜🔗 LangChain Local AI — Practical Examples

> A hands-on notebook exploring LangChain with **local Ollama models** and **Google Gemini**.
> Covers chains, structured output, batch processing, image vision, and multi-step pipelines.

---

## 📋 Table of Contents

1. [Setup & Installation](#1-setup--installation)
2. [Basic Chain — Stream & Batch](#2-basic-chain--stream--batch)
3. [Structured Output with Pydantic](#3-structured-output-with-pydantic)
4. [Invoice Extraction (Single)](#4-invoice-extraction-single)
5. [Invoice Batch Processing](#5-invoice-batch-processing)
6. [Image Vision with Ollama](#6-image-vision-with-ollama)
7. [Multi-Step Pipeline with Gemini](#7-multi-step-pipeline-with-gemini)

---

## Requirements

- [Ollama](https://ollama.com/) installed and running locally
- Models pulled: `ollama pull llama3.2:1b` and `ollama pull moondream`
- A `.env` file with `GOOGLE_API_KEY=...` for the Gemini section


---
## 1. Setup & Installation


In [3]:
# Install required packages (run once)
# !pip install langchain langchain-ollama langchain-google-genai pydantic python-dotenv

In [16]:
# --- Standard Library ---
import base64
import os

# --- Third-Party ---
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Literal, Optional

# --- LangChain Core ---
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent
# --- LangChain Integrations ---
from langchain_ollama import ChatOllama

load_dotenv()
print("✅ All imports loaded successfully.")

✅ All imports loaded successfully.


---
## 2. Basic Chain — Stream & Batch

Build a simple **prompt → LLM → parser** chain using LCEL (LangChain Expression Language).  
Demonstrates both `.stream()` for real-time output and `.batch()` for parallel queries.


In [5]:
# Initialize the local LLM
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0.3,  # Lower = more factual, less creative
)

# Define a simple prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the following question as a helpful assistant.
Question: {input}
Answer:""")

# Build the chain using LCEL pipe syntax
chain = prompt | llm | StrOutputParser()

print("✅ Chain ready.")

✅ Chain ready.


In [6]:
# Stream a response token-by-token
print("--- Streaming Response ---")
for chunk in chain.stream({"input": "Why do parrots talk?"}):
    print(chunk, end="", flush=True)

--- Streaming Response ---
Parrots are known for their ability to mimic human speech and other sounds, and they do so for a variety of reasons. Here are some possible explanations:

1. **Social interaction**: Parrots are highly social birds that live in flocks in the wild. They use vocalizations to communicate with each other, and mimicry is a key part of their social behavior. By mimicking human speech, parrots can interact with their human caregivers and other birds in their flock.
2. **Learning and problem-solving**: Parrots are intelligent birds that are capable of learning and problem-solving. They may mimic human speech to learn new words or to solve problems, such as figuring out how to get to food or water.
3. **Self-expression**: Some parrots may mimic human speech simply because it sounds interesting or fun to them. This is known as "parrot mimicry" and can be a way for parrots to express themselves and have fun.
4. **Communication with humans**: Parrots are highly attuned to

In [7]:
# Batch multiple questions in one call
questions = [
    "What is the capital of France?",
    "What is the name of the largest and most intelligent mammal?",
]

responses = chain.batch(questions)

print("--- Batch Responses ---")
for q, r in zip(questions, responses):
    print(f"\nQ: {q}")
    print(f"A: {r}")

--- Batch Responses ---

Q: What is the capital of France?
A: The capital of France is Paris.

Q: What is the name of the largest and most intelligent mammal?
A: The largest and most intelligent mammal is the blue whale (Balaenoptera musculus). On average, an adult blue whale can grow up to 82 feet (25 meters) in length and weigh around 150-170 tons (136,000-152,000 kilograms). However, the largest blue whale ever recorded was a female that was found in 1947 off the coast of Iceland, which measured around 108 feet (33 meters) in length and weighed an estimated 210 tons (182,000 kilograms).

As for intelligence, blue whales are considered one of the most intelligent animals on Earth. They have a highly developed brain and are known to be able to communicate with each other using a variety of clicks, whistles, and pulses. They have also been observed exhibiting complex behaviors such as hunting cooperatively and even displaying cultural traditions.

It's worth noting that while blue whal

---
## 3. Structured Output with Pydantic

Use `.with_structured_output()` to make the LLM return a validated **Pydantic object**  
instead of raw text. Great for data extraction pipelines.


In [8]:
# Define the output schema
class ProductInfo(BaseModel):
    name: str        = Field(..., description="The name of the product")
    price: float     = Field(..., description="The price of the product in USD")
    category: str    = Field(..., description="The category of the product")
    in_stock: bool   = Field(..., description="Whether the product is currently in stock")

# Bind schema to LLM
llm_structured = ChatOllama(model="llama3.2:1b", temperature=0)
product_llm = llm_structured.with_structured_output(ProductInfo)

# Invoke and get a real Python object
product = product_llm.invoke("What is the price and availability of the latest MacBook Pro 14?")

print(f"Product : {product.name}")
print(f"Price   : ${product.price}")
print(f"Category: {product.category}")
print(f"In Stock: {product.in_stock}")
print(f"\nJSON dump:\n{product.model_dump_json(indent=2)}")

Product : MacBook Pro 14
Price   : $1799.0
Category: Laptop
In Stock: True

JSON dump:
{
  "name": "MacBook Pro 14",
  "price": 1799.0,
  "category": "Laptop",
  "in_stock": true
}


---
## 4. Invoice Extraction (Single)

Extract structured data from raw invoice text using a nested Pydantic schema.  
Demonstrates `Literal` types, optional fields, and computed `@property` values.


In [19]:
# --- Nested Schema Definition ---

class LineItem(BaseModel):
    description: str
    quantity: int   = Field(ge=1)
    unit_price: float = Field(ge=0)

    @property
    def total(self) -> float:
        return self.quantity * self.unit_price


class Invoice(BaseModel):
    invoice_number: str
    date: str
    vendor_name: str
    vendor_address: Optional[str] = None
    items: List[LineItem]
    subtotal: float
    tax: Optional[float] = None
    total: float
    payment_status: Literal["paid", "pending", "overdue"] = "pending"


# --- LLM Setup ---
# json_schema method is most reliable for local models with complex schemas
invoice_llm = ChatOllama(model="llama3.2:1b", temperature=0)
structured_invoice_llm = invoice_llm.with_structured_output(Invoice, method="json_schema")

print("✅ Invoice schema and LLM ready.")

✅ Invoice schema and LLM ready.


In [20]:
invoice_text = """
Invoice #INV-2025-001
Date: January 15, 2025
From: Acme Corp, 123 Business St

Items:
- Widget Pro (5) @ $29.99 each
- Service Fee (1) @ $50.00

Subtotal: $199.95
Tax: $16.00
Total: $215.95

Status: Paid
"""

invoice = structured_invoice_llm.invoke(
    f"Extract all data from this invoice text accurately:\n{invoice_text}"
)

print(f"Invoice Number : {invoice.invoice_number}")
print(f"Vendor         : {invoice.vendor_name}")
print(f"Date           : {invoice.date}")
print(f"Vendor Address : {invoice.vendor_address or 'N/A'}")
print(f"Subtotal       : ${invoice.subtotal:.2f}")
print(f"Tax            : ${invoice.tax or 0:.2f}")
print(f"Total          : ${invoice.total:.2f}")
print(f"Status         : {invoice.payment_status}")
print("\nLine Items:")
for item in invoice.items:
    print(f"  - {item.description}: {item.quantity} x ${item.unit_price:.2f} = ${item.total:.2f}")

Invoice Number : INV-2025-001
Vendor         : Acme Corp
Date           : January 15, 2025
Vendor Address : N/A
Subtotal       : $199.95
Tax            : $16.00
Total          : $215.95
Status         : paid

Line Items:
  - Widget Pro (5): 5 x $29.99 = $149.95
  - Service Fee (1): 1 x $50.00 = $50.00


---
## 5. Invoice Batch Processing

Scale the extraction to process **multiple invoices** in a loop with error handling.


In [11]:
# Simplified schema for batch processing
class InvoiceSummary(BaseModel):
    invoice_number: str
    date: str
    vendor_name: str
    items: List[LineItem]
    total: float
    payment_status: Literal["paid", "pending", "overdue"] = "pending"


batch_llm = ChatOllama(model="llama3.2:1b", temperature=0, format="json")
batch_structured_llm = batch_llm.with_structured_output(InvoiceSummary, method="json_schema")

# Sample invoice texts
invoices_data = [
    "Invoice #INV-001, Date: Jan 10, From: Apple Store. Item: iPhone 15 (1) @ $799. Total: $799. Status: Paid",
    "Invoice #INV-002, Date: Jan 12, From: Amazon. Item: USB-C Cable (2) @ $15. Total: $30. Status: Pending",
    "Invoice #6789, Date: Feb 01, From: Cloud Hosting. Item: Server Pro (1) @ $50. Total: $50. Status: Overdue",
    "Invoice #TX-99, Date: Feb 05, From: Local Cafe. Item: Coffee (10) @ $4.50. Total: $45. Status: Paid",
]

processed_invoices = []

print(f"--- Processing {len(invoices_data)} invoices ---\n")

for i, text in enumerate(invoices_data, 1):
    try:
        print(f"Processing Invoice {i}...")
        result = batch_structured_llm.invoke(
            f"Extract structured data from this invoice text: {text}"
        )
        processed_invoices.append(result)
    except Exception as e:
        print(f"  ⚠️  Error on invoice {i}: {e}")

print("\n--- Results ---")
for inv in processed_invoices:
    status_icon = {"paid": "✅", "pending": "⏳", "overdue": "🔴"}.get(inv.payment_status, "")
    print(f"[{inv.invoice_number}] {inv.vendor_name} — ${inv.total} {status_icon}")

--- Processing 4 invoices ---

Processing Invoice 1...
Processing Invoice 2...
Processing Invoice 3...
Processing Invoice 4...

--- Results ---
[INV-001] Apple Store — $799.0 ✅
[INV-002] Amazon — $30.0 ⏳
[6789] Cloud Hosting — $50.0 🔴
[TX-99] Local Cafe — $45.0 ✅


---
## 6. Image Vision with Ollama

Send images to a local vision model (`moondream`) using Base64 encoding.  
Works fully offline — no API key needed.

> 💡 **Tip:** Replace `IMAGE_PATH` with the path to any image on your machine.


In [12]:
def encode_image(image_path: str) -> str:
    """Convert a local image file to a Base64-encoded string."""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def analyze_image(image_path: str, prompt: str = "What is in this image?") -> str:
    """Send an image to the local Moondream vision model and return the description."""
    vision_llm = ChatOllama(model="moondream")
    image_b64 = encode_image(image_path)

    # Detect format from extension
    ext = image_path.rsplit(".", 1)[-1].lower()
    mime = "image/gif" if ext == "gif" else f"image/{ext}"

    message = HumanMessage(
        content=[
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{image_b64}"}},
        ]
    )
    return vision_llm.invoke([message]).content


print("✅ Vision helper functions defined.")

✅ Vision helper functions defined.


In [13]:
# ✏️  Change this path to a real image on your machine
IMAGE_PATH = "C:\\Users\\hp\\Desktop\\download.jpg"    

if os.path.exists(IMAGE_PATH):
    description = analyze_image(
        IMAGE_PATH,
        prompt="According to what you see, can you explain what is happening in this image step by step?",
    )
    print("--- Model Analysis ---")
    print(description)
else:
    print(f"⚠️  Image not found at '{IMAGE_PATH}'. Update IMAGE_PATH to run this cell.")

--- Model Analysis ---

The image features a small, adorable pig with pink ears and a pink bow on its head. The pig is sitting in a field of yellow leaves, giving the impression that it is in a fall or autumn setting. The pig is looking directly at the camera, creating a sense of connection with the viewer. The background is blurred, drawing focus to the pig and its surroundings.


---
## 7. Multi-Step Pipeline with Gemini

A two-step chain powered by **Google Gemini**:
1. Ask the model to **pick** a well-known movie
2. Ask it to **detail** that movie using the first response as input

Showcases deeply nested schemas and chaining structured calls.

> 🔑 Requires `GOOGLE_API_KEY` in your `.env` file.


In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI

# --- Pydantic Schemas ---

class MovieRef(BaseModel):
    title: str           = Field(..., description="Official movie title")
    year: Optional[int]  = Field(None, description="Release year if known")


class MoviePick(BaseModel):
    movie: MovieRef
    why_known: str       = Field(..., description="Short reason why this movie is well-known")
    confidence: float    = Field(..., ge=0, le=1, description="Confidence score from 0 to 1")


class PersonRole(BaseModel):
    name: str
    role: str = Field(..., description="Character name or production role")


class MovieMeta(BaseModel):
    genres: List[str]
    language: Optional[str] = None
    country: Optional[str] = None
    runtime_minutes: Optional[int] = None


class MovieRatings(BaseModel):
    imdb: Optional[float] = Field(None, ge=0, le=10)
    rotten_tomatoes: Optional[int] = Field(None, ge=0, le=100)


class MovieDetails(BaseModel):
    movie: MovieRef
    plot: str
    themes: List[str]
    key_people: List[PersonRole]
    metadata: MovieMeta
    ratings: MovieRatings


print("✅ Movie schemas defined.")

✅ Movie schemas defined.


In [21]:
# Initialize Gemini
gemini = create_agent(
    model="gemini-3.1-flash-lite-preview",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0,
    response_format="json",
    
)

# Step 1: Pick a movie
pick_llm = gemini.with_structured_output(MoviePick)
movie_pick = pick_llm.invoke(
    "Choose ONE real, famous movie you are very confident you know well. "
    "Return valid structured data only."
)

# Step 2: Get full details for that movie
details_llm = gemini.with_structured_output(MovieDetails)
movie_details = details_llm.invoke(
    f"Provide accurate structured details for this movie: "
    f"{movie_pick.movie.title} ({movie_pick.movie.year})."
)

# --- Print Results ---
print(f"🎬 Picked: {movie_pick.movie.title} ({movie_pick.movie.year})")
print(f"   Confidence: {movie_pick.confidence:.0%}")
print(f"   Why known: {movie_pick.why_known}")

print("\n--- MoviePick JSON ---")
print(movie_pick.model_dump_json(indent=2))

print("\n--- MovieDetails JSON ---")
print(movie_details.model_dump_json(indent=2))

TypeError: create_agent() got an unexpected keyword argument 'google_api_key'

---

## ✅ Summary

| Section | Concept Covered |
|---|---|
| Basic Chain | LCEL, `.stream()`, `.batch()` |
| Structured Output | Pydantic + `.with_structured_output()` |
| Invoice Extraction | Nested schemas, `Literal`, `Optional` |
| Batch Processing | Loops, error handling, `format="json"` |
| Image Vision | Base64 encoding, multimodal messages |
| Multi-Step Gemini | Chaining structured calls, nested models |

---
*Built with [LangChain](https://python.langchain.com/) · [Ollama](https://ollama.com/) · [Google Gemini](https://ai.google.dev/)*
